# Ensemble, xác nhận và tệp nộp

Bước 14–19 của kế hoạch S6E11.

Phân biệt meta CV dùng dự đoán OOF đã lưu với xác nhận lồng (nested CV), nơi mỗi phần giữ lại được dự đoán bằng pipeline học hoàn toàn trong phần huấn luyện tương ứng. Hai họ cây dùng D03 tương ứng để so cách gộp. Khi bỏ một thành viên vẫn đạt ngưỡng sàng lọc và giảm dưới 0.0001 AUC, xác nhận lồng ưu tiên tổ hợp gọn hơn. Các hiện vật của chặng cuối ghi nguồn dự đoán, cách refit và tệp nộp nếu có.

## Chạy lại

Chạy từ thư mục `plan_runs` với môi trường Python đã dùng cho RTX 4060. Trình chạy tiếp tục từ checkpoint có cấu hình khớp. Cell dưới đọc kết quả theo mặc định.

```powershell
& '..\.build\venv\Scripts\python.exe' -B -X utf8 run_plan.py --stage all --compare-xgb-digits --tune-family xgb --seed-anchor XGB-D03 --stack-lgb D03 --stack-xgb XGB-D03
```

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd().resolve()
roots = [Path(os.environ["S6E11_PLAN_RUNS"])] if "S6E11_PLAN_RUNS" in os.environ else []
for base in (here, *here.parents):
    roots.extend((base, base / "plan_runs", base / "s6e11_gpu" / "plan_runs",
                  base / "Predicting Credit Card Fraud" / "s6e11_gpu" / "plan_runs"))
plan_dir = next((path for path in roots if (path / "run_plan.py").is_file()), None)
if plan_dir is None:
    raise FileNotFoundError("Mở notebook trong dự án hoặc đặt S6E11_PLAN_RUNS.")

CHAY_LAI = False
if CHAY_LAI:
    subprocess.run([sys.executable, "-B", "-X", "utf8", str(plan_dir / "run_plan.py"),
                    *['--stage', 'all', '--compare-xgb-digits', '--tune-family', 'xgb', '--seed-anchor', 'XGB-D03', '--stack-lgb', 'D03', '--stack-xgb', 'XGB-D03']],
                   cwd=plan_dir, check=True)

## Kết quả tại thời điểm tạo notebook

| Mã | Đối chứng | ROC AUC OOF | Δ AUC | AP | Log-loss | Giây | Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5 |
| :--- | :--- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| `FINAL` | `XGB-D03` | 0.950345 | +0.000163 | 0.529376 | 0.044347 | — | 0.949878 | 0.951163 | 0.955174 | 0.951022 | 0.944769 |
| `ALL3-MEAN` | `XGB-D03` | 0.950278 | +0.000095 | 0.529063 | 0.044400 | — | 0.949636 | 0.951153 | 0.955191 | 0.951014 | 0.944623 |
| `TREES2-MEAN` | `XGB-D03` | 0.950276 | +0.000094 | 0.528939 | 0.044406 | — | 0.949860 | 0.951070 | 0.955019 | 0.951198 | 0.944514 |
| `ALL3-RANK` | `XGB-D03` | 0.950276 | +0.000093 | 0.528497 | 0.926446 | — | 0.949657 | 0.951243 | 0.955181 | 0.951008 | 0.944529 |
| `TREES2-RANK` | `XGB-D03` | 0.950267 | +0.000085 | 0.528893 | 0.931237 | — | 0.949845 | 0.951051 | 0.955004 | 0.951204 | 0.944512 |
| `XGB-D03-R125` | `XGB-D03` | 0.950146 | -0.000036 | 0.529329 | 0.044384 | 346.8 | 0.949779 | 0.950968 | 0.954931 | 0.950869 | 0.944525 |

Các điểm dưới đây do meta CV trên OOF ghi lại. Đã hiển thị 8 dòng cao nhất khi có đủ dữ liệu. Xem `ensemble/ensemble_metrics.csv`.

| Khảo sát ensemble | Phương pháp | ROC AUC | Δ so mô hình đơn |
|:---|:---|---:|---:|
| `logistic_stack_seed8_c0.1` | `logistic_stack` | 0.950394 | +0.000212 |
| `logistic_stack_seed8_c0.01` | `logistic_stack` | 0.950393 | +0.000211 |
| `rank_logit_stack_seed8_c0.1` | `rank_logit_stack` | 0.950392 | +0.000209 |
| `logistic_stack_seed8_c1` | `logistic_stack` | 0.950390 | +0.000208 |
| `rank_logit_stack_seed8_c0.01` | `rank_logit_stack` | 0.950388 | +0.000206 |
| `rank_logit_stack_seed8_c1` | `rank_logit_stack` | 0.950387 | +0.000204 |
| `logistic_stack_seed7_c0.1` | `logistic_stack` | 0.950386 | +0.000203 |
| `logistic_stack_seed7_c1` | `logistic_stack` | 0.950384 | +0.000202 |

Đã xác minh ROC AUC 0.950345 qua xác nhận lồng trên 5 outer fold. AUC theo fold lần lượt là 0.949878, 0.951163, 0.955174, 0.951022, 0.944769. Các họ mô hình và cấu hình tầng hai được chọn ở giai đoạn khảo sát.

Chọn `NESTED-STACK` với ROC AUC OOF 0.950345. Mô hình đơn đối chứng là `XGB-D03`. Loại điểm `model_probability`. Xem `final/selection.json`.

Bootstrap ghép cặp theo lớp dùng dự đoán cố định. Khoảng dưới đây đo biến thiên lấy mẫu của bộ dự đoán đã lưu.

| Ứng viên cuối | Δ ROC AUC | Fold tăng | Khoảng bootstrap 95% | Qua sàng lọc |
|:---|---:|---:|:---|:---|
| `NESTED-STACK` | +0.000163 | 5 | [+0.000025, +0.000291] | Có |
| `ALL3-MEAN` | +0.000095 | 4 | [-0.000104, +0.000300] | Không |
| `TREES2-MEAN` | +0.000094 | 4 | [+0.000011, +0.000186] | Không |
| `ALL3-RANK` | +0.000093 | 3 | [-0.000105, +0.000298] | Không |
| `SEED3-MEAN` | +0.000087 | 5 | [+0.000008, +0.000168] | Không |
| `TREES2-RANK` | +0.000085 | 3 | [+0.000002, +0.000179] | Không |
| `SEED3-RANK` | +0.000081 | 4 | [+0.000005, +0.000161] | Không |

`submission.csv` qua kiểm tra cột, 299,984 ID, điểm dự đoán và SHA-256 `4ddddd228ce11ace6e0b5e5468632c2f8df92700b4f26310ce4461fbad53788f`.

## Đọc dữ liệu kết quả hiện tại

Cell này xác minh những lượt hoàn tất từ dự đoán OOF và tệp metric.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

sys.path.insert(0, str(plan_dir))
from report_results import final_summary, read_stage

data_dir = plan_dir / "results" / "full"
rows, pending, notes = read_stage('07-ensemble-final', data_dir)
if rows:
    points = [{"mã": row["name"], "đối chứng": row["config"].get("parent"),
               "ROC AUC OOF": row["oof_auc"], "AP": row["average_precision"],
               "delta AUC": row.get("delta_oof_auc"), "log-loss": row["log_loss"],
               **{f"fold {item['fold']}": item["auc"] for item in row["folds"]}}
              for row in rows]
    display(pd.DataFrame(points))
if pending:
    display(Markdown(pending))
if notes:
    display(Markdown("\n".join(f"- {note}" for note in notes)))
if True:
    display(Markdown(final_summary(data_dir, markdown_links=False)))

## Xem cấu hình và log theo fold

Hiển thị lượt có AUC cao nhất trong nhóm hiện tại.

In [ ]:
if rows:
    record = rows[0]
    directory = data_dir / record["name"]
    display(record["config"])
    display(pd.DataFrame([
        {key: item.get(key) for key in
         ("fold", "auc", "device", "seconds", "iterations", "converged", "rounds")}
        for item in record["folds"]
    ]))
    with np.load(directory / "predictions.npz", allow_pickle=False) as saved:
        display({name: saved[name].shape for name in ("ids", "fold", "oof", "test_ids", "test")})

## Phạm vi diễn giải

Điểm khảo sát và điểm xác nhận dùng hai quy trình khác nhau. Tệp nộp được kiểm tra cục bộ và chỉ có điểm Kaggle khi có kết quả nộp thực tế.